# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset package using the `mlcroissant` library. All dataset entities, such as record sets and fields, are consistently referenced via their `@id` identifiers, as required for Croissant compliance.

### Dataset Source

Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and record data from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List all record sets and their fields using @id
print("Record sets available in the dataset:")
record_sets = dataset.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (with @id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis, referencing record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set, referenced by their @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for RecordSet @id {record_set_id}:\n{df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found.\n")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps—filtering records, normalizing numeric fields, and grouping data—for an example record set (using the record set and numeric field `@id` identified above). All references use `@id` notation.

In [ ]:
# EDA: Choose a record set and numeric field for demonstration

# Here, we'll select the first record set (if available) and a numeric field within it.
if len(dataframes):
    chosen_record_set_id = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set_id]
    print(f"Using RecordSet @id: {chosen_record_set_id}")
    # Find numeric fields (type = 'Number', 'Integer', 'Float')
    # This requires accessing the record set schema
    numeric_types = ["Number", "Integer", "Float"]
    numeric_field_id = None
    group_field_id = None
    fields = [x for x in dataset.record_sets if x.id == chosen_record_set_id][0].fields
    for field in fields:
        if field.data_type in numeric_types:
            numeric_field_id = field.id
            break
    for field in fields:
        if field.data_type in ["Text", "String"] and field.id != numeric_field_id:
            group_field_id = field.id
            break
    if numeric_field_id:
        col = numeric_field_id
        print(f"\nNumeric field selected (by @id): {numeric_field_id}")
        print(f"Group field candidate (by @id): {group_field_id}")
        # Perform filtering
        threshold = 10
        if col in df.columns:
            filtered_df = df[pd.to_numeric(df[col], errors='coerce') > threshold]
            print(f"\nRecords with {col} > {threshold}:")
            print(filtered_df.head())
            # Normalization
            filtered_df = filtered_df.copy()
            filtered_df[f"{col}_normalized"] = (pd.to_numeric(filtered_df[col], errors='coerce') - pd.to_numeric(filtered_df[col], errors='coerce').mean()) / pd.to_numeric(filtered_df[col], errors='coerce').std()
            print(f"\nNormalized column {col} for filtered records:")
            print(filtered_df[[col, f"{col}_normalized"]].head())
            # Grouping
            if group_field_id and group_field_id in filtered_df.columns:
                gdf = filtered_df.groupby(group_field_id)[col].mean().reset_index()
                print(f"\nGrouped mean of {col} by {group_field_id}:")
                print(gdf.head())
            else:
                print("No available group field for aggregation.")
        else:
            print(f"Numeric field {col} not present in columns.")
    else:
        print("No numeric field found for this record set.")
else:
    print("No dataframes loaded for analysis.")

## 5. Visualization

Visualize data distributions or relationships using field `@id`s. Here is an example histogram of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field (using @id)
if 'filtered_df' in locals() and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric data found for visualization.")

## 6. Conclusion

In this notebook, we have loaded and explored the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library. All access and analysis steps were explicitly referenced by Croissant entity `@id`, including record sets and fields. You can adapt this workflow for more detailed analyses or for use with other Croissant-compliant datasets.